In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder\
        .appName('Spark_Skew_Salt')\
        .master('local[2]')\
        .getOrCreate()
# =============================================
# 1. 构造 100 条数据 → 热点多（10个热点key，每个10条 = 各占10%）
# =============================================
data=[]
# 10个热点key，每个10条数据 → 共100条
hot_keys = [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010]
for key in hot_keys:
    for i in range(10):
        data.append((key,'user_data'))
df_big=spark.createDataFrame(data,['join_key','info'])
print("大表总条数:", df_big.count())  # 100条
df_big.show()



大表总条数: 100
+--------+---------+
|join_key|     info|
+--------+---------+
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1001|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
|    1002|user_data|
+--------+---------+
only showing top 20 rows



In [2]:
# 第 1 步：查看热点分布（确认热点多）
df_big.groupBy('join_key').count().orderBy(desc('count')).show()
# 结果（10 个热点，每个 10 条 → 无法抽取！必须加盐）

+--------+-----+
|join_key|count|
+--------+-----+
|    1002|   10|
|    1005|   10|
|    1001|   10|
|    1004|   10|
|    1003|   10|
|    1010|   10|
|    1009|   10|
|    1007|   10|
|    1008|   10|
|    1006|   10|
+--------+-----+



In [3]:
# 第 2 步：构造小表（维度表）
dim_data=[(k,f'类型_{k}') for k in hot_keys]
df_small=spark.createDataFrame(dim_data,['join_key','type'])
print('维度表的数据量:',df_small.count())
df_small.show()


维度表的数据量: 10
+--------+---------+
|join_key|     type|
+--------+---------+
|    1001|类型_1001|
|    1002|类型_1002|
|    1003|类型_1003|
|    1004|类型_1004|
|    1005|类型_1005|
|    1006|类型_1006|
|    1007|类型_1007|
|    1008|类型_1008|
|    1009|类型_1009|
|    1010|类型_1010|
+--------+---------+



In [4]:
# 第 3 步：【核心】大表加盐（打散热点）
df_big_salt=df_big.withColumn('salt',(rand()*3).cast('int'))
# df_big_salt=df_big.withColumn('salt',(rand()*3))
df_big_salt.show(20)

+--------+---------+----+
|join_key|     info|salt|
+--------+---------+----+
|    1001|user_data|   2|
|    1001|user_data|   2|
|    1001|user_data|   2|
|    1001|user_data|   2|
|    1001|user_data|   1|
|    1001|user_data|   0|
|    1001|user_data|   2|
|    1001|user_data|   2|
|    1001|user_data|   2|
|    1001|user_data|   1|
|    1002|user_data|   1|
|    1002|user_data|   1|
|    1002|user_data|   1|
|    1002|user_data|   0|
|    1002|user_data|   2|
|    1002|user_data|   0|
|    1002|user_data|   1|
|    1002|user_data|   1|
|    1002|user_data|   2|
|    1002|user_data|   1|
+--------+---------+----+
only showing top 20 rows



In [5]:
# 生成新的JOIN键：原key_随机数
df_big_salt=df_big_salt.withColumn('new_join_key',
                                   concat(col('join_key'),
                                          lit('_'),col('salt')))
df_big_salt.show()
# 盐后数据样子（热点被自动打散）
# 10条数据变成 3个key分担

+--------+---------+----+------------+
|join_key|     info|salt|new_join_key|
+--------+---------+----+------------+
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   1|      1001_1|
|    1001|user_data|   0|      1001_0|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   1|      1001_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   0|      1002_0|
|    1002|user_data|   2|      1002_2|
|    1002|user_data|   0|      1002_0|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   2|      1002_2|
|    1002|user_data|   1|      1002_1|
+--------+---------+----+------------+
only showing top 20 rows



In [6]:
# 1. 构建盐值表（决定拆成几份：0,1,2 → 3份）
df_salt=spark.createDataFrame([(0,),(1,),(2,)],['salt'])
df_salt.show()

+----+
|salt|
+----+
|   0|
|   1|
|   2|
+----+



In [7]:
# 1. 构建盐值表（决定拆成几份：0,1,2 → 3份）
df_salt = spark.range(3).toDF("salt")  # 3 行：0,1,2
df_salt.show()


+----+
|salt|
+----+
|   0|
|   1|
|   2|
+----+



In [8]:
df_small.show()

+--------+---------+
|join_key|     type|
+--------+---------+
|    1001|类型_1001|
|    1002|类型_1002|
|    1003|类型_1003|
|    1004|类型_1004|
|    1005|类型_1005|
|    1006|类型_1006|
|    1007|类型_1007|
|    1008|类型_1008|
|    1009|类型_1009|
|    1010|类型_1010|
+--------+---------+



In [ ]:
# 2. crossJoin 直接扩容
df_small_expand=df_small.crossJoin(df_salt)
print('小表crossJoin 扩容后:')
df_small_expand.show(30,truncate=False)


小表crossJoin 扩容后:
+--------+---------+----+
|join_key|type     |salt|
+--------+---------+----+
|1001    |类型_1001|0   |
|1001    |类型_1001|1   |
|1001    |类型_1001|2   |
|1002    |类型_1002|0   |
|1002    |类型_1002|1   |
|1002    |类型_1002|2   |
|1003    |类型_1003|0   |
|1003    |类型_1003|1   |
|1003    |类型_1003|2   |
|1004    |类型_1004|0   |
|1004    |类型_1004|1   |
|1004    |类型_1004|2   |
|1005    |类型_1005|0   |
|1005    |类型_1005|1   |
|1005    |类型_1005|2   |
|1006    |类型_1006|0   |
|1006    |类型_1006|1   |
|1006    |类型_1006|2   |
|1007    |类型_1007|0   |
|1007    |类型_1007|1   |
|1007    |类型_1007|2   |
|1008    |类型_1008|0   |
|1008    |类型_1008|1   |
|1008    |类型_1008|2   |
|1009    |类型_1009|0   |
|1009    |类型_1009|1   |
|1009    |类型_1009|2   |
|1010    |类型_1010|0   |
|1010    |类型_1010|1   |
|1010    |类型_1010|2   |
+--------+---------+----+



In [10]:
# 3. 生成新关联key
df_small_expand=df_small_expand.withColumn(
    'new_join_key',
    concat(col('join_key'),lit('_'),col('salt'))
)
print("\n=== 小表 crossJoin 扩容后效果 ===")
df_small_expand.show(30,truncate=False)


=== 小表 crossJoin 扩容后效果 ===
+--------+---------+----+------------+
|join_key|type     |salt|new_join_key|
+--------+---------+----+------------+
|1001    |类型_1001|0   |1001_0      |
|1001    |类型_1001|1   |1001_1      |
|1001    |类型_1001|2   |1001_2      |
|1002    |类型_1002|0   |1002_0      |
|1002    |类型_1002|1   |1002_1      |
|1002    |类型_1002|2   |1002_2      |
|1003    |类型_1003|0   |1003_0      |
|1003    |类型_1003|1   |1003_1      |
|1003    |类型_1003|2   |1003_2      |
|1004    |类型_1004|0   |1004_0      |
|1004    |类型_1004|1   |1004_1      |
|1004    |类型_1004|2   |1004_2      |
|1005    |类型_1005|0   |1005_0      |
|1005    |类型_1005|1   |1005_1      |
|1005    |类型_1005|2   |1005_2      |
|1006    |类型_1006|0   |1006_0      |
|1006    |类型_1006|1   |1006_1      |
|1006    |类型_1006|2   |1006_2      |
|1007    |类型_1007|0   |1007_0      |
|1007    |类型_1007|1   |1007_1      |
|1007    |类型_1007|2   |1007_2      |
|1008    |类型_1008|0   |1008_0      |
|1008    |类型_1008|1   |1008_1      |
|100

In [11]:
# ======================
# 6.  最终正确 JOIN
# ======================
df_big_salt.show()


+--------+---------+----+------------+
|join_key|     info|salt|new_join_key|
+--------+---------+----+------------+
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   1|      1001_1|
|    1001|user_data|   0|      1001_0|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   2|      1001_2|
|    1001|user_data|   1|      1001_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   0|      1002_0|
|    1002|user_data|   2|      1002_2|
|    1002|user_data|   0|      1002_0|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   1|      1002_1|
|    1002|user_data|   2|      1002_2|
|    1002|user_data|   1|      1002_1|
+--------+---------+----+------------+
only showing top 20 rows



In [12]:
df_small_expand.show()

+--------+---------+----+------------+
|join_key|     type|salt|new_join_key|
+--------+---------+----+------------+
|    1001|类型_1001|   0|      1001_0|
|    1001|类型_1001|   1|      1001_1|
|    1001|类型_1001|   2|      1001_2|
|    1002|类型_1002|   0|      1002_0|
|    1002|类型_1002|   1|      1002_1|
|    1002|类型_1002|   2|      1002_2|
|    1003|类型_1003|   0|      1003_0|
|    1003|类型_1003|   1|      1003_1|
|    1003|类型_1003|   2|      1003_2|
|    1004|类型_1004|   0|      1004_0|
|    1004|类型_1004|   1|      1004_1|
|    1004|类型_1004|   2|      1004_2|
|    1005|类型_1005|   0|      1005_0|
|    1005|类型_1005|   1|      1005_1|
|    1005|类型_1005|   2|      1005_2|
|    1006|类型_1006|   0|      1006_0|
|    1006|类型_1006|   1|      1006_1|
|    1006|类型_1006|   2|      1006_2|
|    1007|类型_1007|   0|      1007_0|
|    1007|类型_1007|   1|      1007_1|
+--------+---------+----+------------+
only showing top 20 rows



In [28]:
# 小表扩容后
df_small_expand = df_small_expand.drop("join_key")  # 删除原始 key

In [29]:
# 7:这里有两种关联的方式
# 1：加盐的大表 df_big_salt 直接join 扩容的小表 df_small_expand
# 2：加盐的大表df_big_salt广播broadcastk扩容后的小表df_small_expand

# 加盐的大表df_big_salt广播broadcastk扩容后的小表df_small_expand
df_result=df_big_salt.join(
    broadcast(df_small_expand),
    df_big_salt['new_join_key']== df_small_expand['new_join_key'])
print(df_result.count())
df_result.show()


100
+--------+---------+----+------------+---------+----+------------+
|join_key|     info|salt|new_join_key|     type|salt|new_join_key|
+--------+---------+----+------------+---------+----+------------+
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   1|      1001_1|类型_1001|   1|      1001_1|
|    1001|user_data|   0|      1001_0|类型_1001|   0|      1001_0|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   1|      1001_1|类型_1001|   1|      1001_1|
|    1002|user_data|   1|      1002_1|类型_1002|   1|      1002_1|
|    1002|user_data|   1|      1002_1|类型_1002|   1|      1002_1|
|    1002|user_

In [30]:
# 第一种方法加盐的大表 df_big_salt 直接join 扩容的小表 df_small_expand
df_result1=df_big_salt.join(df_small_expand,
                            df_big_salt['new_join_key']== df_small_expand['new_join_key'])
print(df_result1.count())
df_result1.show()

100
+--------+---------+----+------------+---------+----+------------+
|join_key|     info|salt|new_join_key|     type|salt|new_join_key|
+--------+---------+----+------------+---------+----+------------+
|    1001|user_data|   0|      1001_0|类型_1001|   0|      1001_0|
|    1001|user_data|   1|      1001_1|类型_1001|   1|      1001_1|
|    1001|user_data|   1|      1001_1|类型_1001|   1|      1001_1|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1001|user_data|   2|      1001_2|类型_1001|   2|      1001_2|
|    1002|user_data|   0|      1002_0|类型_1002|   0|      1002_0|
|    1002|user_data|   0|      1002_0|类型_1002|   0|      1002_0|
|    1002|user_

In [31]:
# ======================
# 7. 去盐 + 聚合（恢复原始数据）
# ======================
df_final1=df_result1.select(col('join_key'),
                          col('info'),
                          col('type'))
df_final1.show()

+--------+---------+---------+
|join_key|     info|     type|
+--------+---------+---------+
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
+--------+---------+---------+
only showing top 20 rows



In [32]:
# ======================
# 7. 去盐 + 聚合（恢复原始数据）
# ======================
df_final=df_result.select(col('join_key'),
                          col('info'),
                          col('type'))
df_final.show()

+--------+---------+---------+
|join_key|     info|     type|
+--------+---------+---------+
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1001|user_data|类型_1001|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
|    1002|user_data|类型_1002|
+--------+---------+---------+
only showing top 20 rows



In [34]:
print("\n✅ JOIN + 去盐后总条数：", df_final.count())  # 100 ✅
print("\n✅ JOIN + 去盐后总条数：", df_final1.count())  # 100 ✅


✅ JOIN + 去盐后总条数： 100

✅ JOIN + 去盐后总条数： 100


In [35]:
df_final.groupBy("join_key", "type") \
        .count() \
        .orderBy("join_key") \
        .show(truncate=False)

+--------+---------+-----+
|join_key|type     |count|
+--------+---------+-----+
|1001    |类型_1001|10   |
|1002    |类型_1002|10   |
|1003    |类型_1003|10   |
|1004    |类型_1004|10   |
|1005    |类型_1005|10   |
|1006    |类型_1006|10   |
|1007    |类型_1007|10   |
|1008    |类型_1008|10   |
|1009    |类型_1009|10   |
|1010    |类型_1010|10   |
+--------+---------+-----+



In [36]:
spark.stop()